# 03 Network

The main spec uses `MIN_TX = 2`; `MIN_TX = 1` gives the sensitivity tables. Figures use the main spec.

Build the firm x product bipartite network from PQR and project it onto products.

Proximity between products i and j follows Hidalgo et al. (2007):
    phi_ij = C_ij / max(k_i, k_j)
where C_ij = number of firms that supplied both products and k_i = number of firms that
supplied product i. This equals min(P(i|j), P(j|i)) and keeps large products from
connecting to everything.

A firm "supplies" a product if it has at least MIN_TX PQR transactions for it, to drop
one-off purchases. Outputs go to data/processed/network/.


Community detection, null-model test, and centrality on the product network.

Steps
1. Louvain communities on the phi-weighted product network (best of 50 seeds; stability = mean
   ARI between seed runs).
2. Null model: rewire the firm x product bipartite network keeping every firm's and product's
   degree fixed (random edge swaps), re-project, re-run Louvain. Compare observed modularity
   and observed alignment with disease areas against 200 null draws.
3. Alignment of communities with official disease areas (NMI, ARI, cross-tab).
4. Centrality: strength (sum of phi), betweenness (distance = 1 - phi), eigenvector.


Figures for the product network (main spec MIN_TX=2).
f5: product space backbone (maximum spanning tree + strong ties), node color = disease area,
    shaded regions = Louvain communities, node size = spend.
f6: null-model test (observed vs 200 degree-preserving rewirings).
f7: disease-area composition of each community.

In [ ]:
%run 00_utils.ipynb

import numpy as np, pandas as pd, networkx as nx, matplotlib
import matplotlib.pyplot as plt
from networkx.algorithms.community import louvain_communities, modularity
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

## Build

In [ ]:
def build_network(MIN_TX):
    OUT = ROOT / "data" / "processed" / "network"; OUT.mkdir(parents=True, exist_ok=True)

    pqr = pd.read_csv(ROOT / "data" / "processed" / "pqr_medicines_clean.csv")

    def bipartite_edges(df, min_tx=MIN_TX):
        e = df.groupby(["firm", "product"]).agg(n_tx=("pk", "size"), spend_usd=("cost_usd", "sum"),
                                                first_year=("year", "min"), last_year=("year", "max")).reset_index()
        return e[e.n_tx >= min_tx].reset_index(drop=True)

    e = bipartite_edges(pqr)
    e.to_csv(OUT / f"bipartite_edges_min{MIN_TX}.csv", index=False)

    prod = pqr.groupby("product").agg(area=("area", "first"), spend_usd=("cost_usd", "sum"),
                                      n_tx=("pk", "size"), n_countries=("country", "nunique"),
                                      first_year=("year", "min")).reset_index()
    prod["n_firms"] = prod["product"].map(e.groupby("product")["firm"].nunique()).fillna(0).astype(int)
    firm = pqr.groupby("firm").agg(group=("group", "first"), hq_country=("hq_country", "first"),
                                   spend_usd=("cost_usd", "sum")).reset_index()
    firm["n_products"] = firm["firm"].map(e.groupby("firm")["product"].nunique()).fillna(0).astype(int)
    firm["n_areas"] = firm["firm"].map(e.merge(prod[["product", "area"]]).groupby("firm")["area"].nunique()).fillna(0).astype(int)

    pe = project_products(e)
    pe.to_csv(OUT / f"product_edges_min{MIN_TX}.csv", index=False)
    prod.to_csv(OUT / f"product_nodes_min{MIN_TX}.csv", index=False)
    firm.to_csv(OUT / f"firm_nodes_min{MIN_TX}.csv", index=False)

    G = nx.Graph()
    G.add_nodes_from(prod[prod.n_firms > 0]["product"])
    G.add_weighted_edges_from(pe[["source", "target", "phi"]].itertuples(index=False))
    iso = [n for n in G if G.degree(n) == 0]
    print(f"MIN_TX={MIN_TX}")
    print(f"bipartite: {e.firm.nunique()} firms, {e['product'].nunique()} products, {len(e)} edges "
          f"(dropped {pqr.groupby(['firm','product']).ngroups - len(e)} firm-product pairs with < {MIN_TX} tx)")
    print(f"firms supplying 1 product: {(firm.n_products == 1).sum()} | >=10 products: {(firm.n_products >= 10).sum()}")
    print(f"product network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
          f"density {nx.density(G):.3f}, isolates {len(iso)}, components {nx.number_connected_components(G)}")
    print("phi quantiles:", pe.phi.quantile([.25, .5, .75, .9]).round(2).to_dict())
    print("isolated products:", iso[:15])

## Analysis

In [ ]:
def analyze_network(MIN_TX, N_NULL=200):
    NET = ROOT / "data" / "processed" / "network"
    TAB = ROOT / "outputs" / "tables"
    SUF = "" if MIN_TX == 2 else f"_min{MIN_TX}"   # main spec is MIN_TX=2; others are sensitivity runs
    rng = np.random.default_rng(5205)

    e = pd.read_csv(NET / f"bipartite_edges_min{MIN_TX}.csv")
    nodes = pd.read_csv(NET / f"product_nodes_min{MIN_TX}.csv").set_index("product")
    pe = pd.read_csv(NET / f"product_edges_min{MIN_TX}.csv")

    def graph(pe, prods):
        G = nx.Graph(); G.add_nodes_from(prods)
        G.add_weighted_edges_from(pe[["source", "target", "phi"]].itertuples(index=False))
        return G

    def best_louvain(G, seeds):
        best, runs = None, []
        for s in seeds:
            c = louvain_communities(G, weight="weight", seed=int(s))
            q = modularity(G, c, weight="weight"); runs.append(c)
            if best is None or q > best[1]:
                best = (c, q)
        return best, runs

    def labels(comms, order):
        m = {n: i for i, c in enumerate(comms) for n in c}
        return np.array([m[n] for n in order])

    prods = sorted(e["product"].unique())
    G = graph(pe, prods)
    (comms, Q), runs = best_louvain(G, range(50))
    # order communities by size, name them C1..Ck
    comms = sorted(comms, key=len, reverse=True)
    lab = labels(comms, prods)
    area = nodes.loc[prods, "area"].values
    stab = np.mean([adjusted_rand_score(labels(runs[i], prods), labels(runs[j], prods))
                    for i in range(10) for j in range(i + 1, 10)])
    nmi, ari = normalized_mutual_info_score(area, lab), adjusted_rand_score(area, lab)
    print(f"observed: {len(comms)} communities, modularity Q={Q:.3f}, seed stability ARI={stab:.2f}")
    print(f"alignment with disease area: NMI={nmi:.2f}, ARI={ari:.2f}")

    # ---- null model: degree-preserving bipartite rewiring
    def rewire(edges, n_swaps):
        f = edges["firm"].to_numpy().copy(); p = edges["product"].to_numpy().copy()
        have = set(zip(f, p)); m = len(f)
        for _ in range(n_swaps):
            i, j = rng.integers(m, size=2)
            if f[i] == f[j] or p[i] == p[j]: continue
            if (f[i], p[j]) in have or (f[j], p[i]) in have: continue
            have -= {(f[i], p[i]), (f[j], p[j])}; p[i], p[j] = p[j], p[i]
            have |= {(f[i], p[i]), (f[j], p[j])}
        return pd.DataFrame({"firm": f, "product": p})

    null = []
    for r in range(N_NULL):
        en = rewire(e[["firm", "product"]], 10 * len(e))
        Gn = graph(project_products(en), prods)
        (cn, qn), _ = best_louvain(Gn, rng.integers(1e6, size=3))
        ln = labels(cn, prods)
        null.append((qn, len(cn), normalized_mutual_info_score(area, ln)))
    null = pd.DataFrame(null, columns=["Q", "k", "nmi"])
    null.to_csv(NET / f"null_model_min{MIN_TX}.csv", index=False)
    zQ = (Q - null.Q.mean()) / null.Q.std(); zN = (nmi - null.nmi.mean()) / null.nmi.std()
    pQ = (null.Q >= Q).mean(); pN = (null.nmi >= nmi).mean()
    print(f"null ({N_NULL} rewirings): Q mean {null.Q.mean():.3f} sd {null.Q.std():.3f} -> z={zQ:.1f}, p={pQ:.3f}")
    print(f"null NMI mean {null.nmi.mean():.2f} sd {null.nmi.std():.2f} -> z={zN:.1f}, p={pN:.3f}")

    # ---- centrality
    dist = {(u, v): 1 - d["weight"] + 1e-6 for u, v, d in G.edges(data=True)}
    nx.set_edge_attributes(G, dist, "dist")
    cent = pd.DataFrame({
        "community": [f"C{l+1}" for l in lab],
        "strength": pd.Series(dict(G.degree(weight="weight")))[prods].values,
        "degree": pd.Series(dict(G.degree()))[prods].values,
        "betweenness": pd.Series(nx.betweenness_centrality(G, weight="dist"))[prods].values,
        "eigenvector": pd.Series(nx.eigenvector_centrality_numpy(
            G.subgraph(max(nx.connected_components(G), key=len)), weight="weight")).reindex(prods).fillna(0).values,
    }, index=prods)
    pn = nodes.join(cent, how="inner")
    pn["core"] = pn.strength >= pn.strength.quantile(0.75)
    pn.round(4).to_csv(NET / f"product_nodes_analyzed_min{MIN_TX}.csv")

    # ---- tables
    ct = pd.crosstab(pn.community, pn.area)
    ct["n_products"] = ct.sum(axis=1)
    ct["spend_usd_m"] = pn.groupby("community").spend_usd.sum().div(1e6).round(0)
    ct["median_firms"] = pn.groupby("community").n_firms.median()
    ct["top_products"] = pn.sort_values("spend_usd", ascending=False).groupby("community").apply(
        lambda d: "; ".join(d.index[:3]), include_groups=False)
    ef = e.merge(pn[["community"]], left_on="product", right_index=True)
    ct["top_firms"] = ef.groupby(["community", "firm"]).spend_usd.sum().reset_index().sort_values(
        "spend_usd", ascending=False).groupby("community").apply(lambda d: "; ".join(d.firm[:3]), include_groups=False)
    ct = ct.sort_index(key=lambda s: s.str[1:].astype(int))
    ct.to_csv(TAB / f"t7_communities{SUF}.csv")
    pd.set_option("display.width", 250, "display.max_colwidth", 70)
    print("\n", ct.to_string())

    summ = pd.DataFrame([
        ("products", len(prods)), ("edges", G.number_of_edges()), ("density", round(nx.density(G), 3)),
        ("communities", len(comms)), ("modularity_Q", round(Q, 3)), ("seed_stability_ARI", round(stab, 2)),
        ("null_Q_mean", round(null.Q.mean(), 3)), ("null_Q_sd", round(null.Q.std(), 3)), ("z_Q", round(zQ, 1)), ("p_Q", pQ),
        ("NMI_area", round(nmi, 2)), ("ARI_area", round(ari, 2)), ("null_NMI_mean", round(null.nmi.mean(), 2)),
        ("z_NMI", round(zN, 1)), ("p_NMI", pN)], columns=["metric", "value"])
    summ.to_csv(TAB / f"t8_network_summary{SUF}.csv", index=False)

    top = pn.sort_values("strength", ascending=False)
    top[["area", "community", "n_firms", "strength", "betweenness", "spend_usd"]].head(15).round(3).to_csv(TAB / f"t9_central_products{SUF}.csv")
    print("\nmost central products:\n", top[["area", "community", "n_firms", "strength", "betweenness"]].head(10).round(2).to_string())
    print("\ncorr(strength, n_firms) =", round(pn.strength.corr(pn.n_firms, method="spearman"), 2),
          "| corr(strength, spend) =", round(pn.strength.corr(pn.spend_usd, method="spearman"), 2))
    print("core products by area:", pn[pn.core].area.value_counts().to_dict())
    print("median n_firms core vs periphery:", pn.groupby("core").n_firms.median().to_dict())

## Main spec

In [ ]:
build_network(2)
analyze_network(2)

With 2+ transactions required for a link, the network had 97 firms, 122 products and 491 firm-product links. The product network had 2,876 weighted edges and a density of 0.39.

Louvain found 7 communities (Q = 0.31), and they were stable across seeds (mean ARI 0.87). Modularity in the 200 rewired networks averaged 0.22 (sd 0.018), so the observed value was well above chance (z = 5.2). The rewiring keeps each firm's and product's number of links, so this structure did not come from firm size alone.

Communities matched disease areas only partly. NMI was 0.49, compared with a null mean of 0.07 (z = 19.5).

Core products, meaning the top quartile by network strength, were all HIV (18) or TB (13), with no malaria. They had a median of 7 suppliers, compared with 2 in the periphery. Strength correlated with the number of suppliers (Spearman 0.65) and with spend (0.55).

## Sensitivity

In [ ]:
build_network(1)
analyze_network(1)

When I counted a single transaction as a link, the network kept 7 communities and the same modularity (Q = 0.31, z = 5.8). NMI with disease area fell to 0.36, which makes sense because one-off purchases add links across areas. I kept 2+ transactions as the main spec because the community structure held and these one-off links were dropped.

## Figures

In [ ]:
set_style(grid=False)

NET, F = ROOT / "data" / "processed" / "network", ROOT / "outputs" / "figures"
INK, INK2, MUTED, GRID = "#0b0b0b", "#595959", "#8a8984", "#e6e5e1"
AREA = {"HIV/ARV": "#003D78", "Tuberculosis": "#117996", "Malaria": "#F4B71B"}

def title(ax, t, sub):
    ax.set_title(t, loc="left", fontsize=12, fontweight="bold", color=INK, pad=22)
    ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=9, color=INK2, va="bottom")

nodes = pd.read_csv(NET / "product_nodes_analyzed_min2.csv", index_col=0)
pe = pd.read_csv(NET / "product_edges_min2.csv")
G = nx.Graph(); G.add_nodes_from(nodes.index)
G.add_weighted_edges_from(pe[["source", "target", "phi"]].itertuples(index=False))

# backbone: maximum spanning tree (per component) + all ties with phi >= 0.5
B = nx.maximum_spanning_tree(G, weight="weight")
B.add_edges_from((u, v, d) for u, v, d in G.edges(data=True) if d["weight"] >= 0.5)
# community layout: lay out each community on its own, then place the blocks side by side,
# so the shaded regions are faithful and node color (disease area) shows misfits inside each block.
ANCHOR = {"C1": (0.0, 0.0, 1.0), "C2": (2.75, 0.0, 1.0), "C3": (0.9, -2.55, 0.8),
          "C4": (3.0, -2.2, 0.4), "C5": (3.95, -3.05, 0.1), "C6": (4.35, -3.05, 0.04), "C7": (4.55, -3.05, 0.04)}
pos = {}
for c, (x0, y0, sc) in ANCHOR.items():
    members = nodes.index[nodes.community == c]
    if len(members) > 1:
        H = nx.complete_graph(members)                      # distance = 1 - phi (1.2 if no tie)
        for u, v in H.edges:
            H[u][v]["d"] = 1.05 - G[u][v]["weight"] if G.has_edge(u, v) else 1.2
        p = nx.kamada_kawai_layout(H, weight="d")
    else:
        p = {members[0]: np.zeros(2)}
    arr = np.array(list(p.values())); arr = arr - np.median(arr, 0)
    r = np.quantile(np.abs(arr), 0.95) or 1
    arr = np.clip(arr / r, -1, 1)
    for n, xy in zip(p, arr * sc):
        pos[n] = xy + np.array([x0, y0])

COMM_NAME = {"C1": "C1  HIV generics core", "C2": "C2  TB generics",
             "C3": "C3  Malaria, injectables & originators", "C4": "C4  TB patient kits",
             "C5": "C5-C7  single-supplier\nproducts"}
fig, ax = plt.subplots(figsize=(10, 8.6))
for c, (x0, y0, sc) in ANCHOR.items():
    if c in ("C6", "C7"): continue
    w = sc + 0.18 if c != "C5" else 0.5
    xc = x0 if c != "C5" else 4.25
    ax.add_patch(matplotlib.patches.FancyBboxPatch((xc - w, y0 - sc - 0.18), 2 * w, 2 * sc + 0.36,
                 boxstyle="round,pad=0.02,rounding_size=0.12", fc="#f4f3ef", ec="#d6d5cf", lw=0.8, zorder=0))
    ax.text(xc - w + 0.05, y0 + sc + 0.22, COMM_NAME[c], fontsize=9.5, fontweight="bold", color=INK2, va="bottom")
cross = [(u, v, d) for u, v, d in B.edges(data=True) if nodes.community[u] != nodes.community[v]]
inner = [(u, v, d) for u, v, d in G.edges(data=True) if nodes.community[u] == nodes.community[v] and d["weight"] >= 0.25]
for u, v, d in cross:
    ax.plot(*zip(pos[u], pos[v]), color="#c9c8c2", lw=0.6, zorder=1, alpha=0.6, ls=":")
for u, v, d in inner:
    ax.plot(*zip(pos[u], pos[v]), color="#b9b8b2", lw=0.3 + 1.4 * d["weight"], zorder=1, alpha=0.55)
size = 18 + 700 * np.sqrt(nodes.spend_usd / nodes.spend_usd.max())
ax.scatter([pos[n][0] for n in nodes.index], [pos[n][1] for n in nodes.index], s=size,
           c=nodes.area.map(AREA), edgecolors="white", linewidths=1, zorder=2)
short = lambda s: s.replace(" - FDC", "").replace("Dolutegravir", "DTG").replace("Lamivudine", "3TC") \
    .replace("Tenofovir DF", "TDF").replace("Tenofovir", "TDF").replace("Efavirenz", "EFV") \
    .replace("Zidovudine", "AZT").replace("Nevirapine", "NVP").replace("Emtricitabine", "FTC") \
    .replace("Ethambutol+Isoniazid+Pyrazinamide+Rifampicin (RHZE)", "RHZE").replace("Isoniazid", "INH") \
    .replace("Rifampicin", "RIF").replace(" + ", "+")[:30]
lab = (nodes.sort_values("spend_usd", ascending=False).groupby("community").head(2).index.tolist()
       + ["Artemether+Lumefantrine - FDC", "Moxifloxacin", "Atazanavir+Ritonavir - FDC", "Amikacin"])
for i, n in enumerate(dict.fromkeys(lab)):
    if nodes.community[n] in ("C5", "C6", "C7"): continue
    x, y = pos[n]
    ax.annotate(short(n), (x, y), xytext=(7, 5 if i % 2 else -9), textcoords="offset points", fontsize=7.5,
                color=INK, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85), zorder=3)
for a, col in AREA.items():
    ax.scatter([], [], s=60, c=col, label=a, edgecolors="white")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0.0, 0.02), fontsize=9,
          title="Node color: disease area", title_fontsize=8.5)
ax.set_aspect("equal"); ax.set_axis_off()
title(ax, "The product space of Global Fund medicines",
      "Boxes = network communities (Louvain). Products are close when the same firms supply both; node size = spend")
fig.text(0.02, 0.02, "Solid lines: ties within a community with proximity >= 0.25; dotted: strongest ties between communities (maximum spanning tree).\n"
         "122 products, 97 firms; a firm-product link requires 2+ PQR transactions. Source: Global Fund PQR, 2005-2026.", fontsize=7.5, color=MUTED)
fig.savefig(F / "f5_product_space.png"); plt.show()

The main communities were:

- C1 HIV generics core: 44 products, 86% HIV, USD 9.0 bn (Mylan, Hetero, Cipla)
- C2 TB generics: 43 products, 93% TB, USD 1.9 bn (Macleods, Lupin, Mylan)
- C3 Malaria, injectables and originators: 24 products, 67% malaria, USD 0.7 bn (Guilin, Sanofi, Ipca)
- C4 TB patient kits: 7 products (Svizera, Labesfal, B. Braun)
- C5-C7: fosamprenavir and maraviroc, arteether, and pyronaridine-artesunate, each with very few suppliers

I chose these labels from each community's main disease area and top suppliers, so they are my interpretation.

The misfits were the most informative part. Artemether-lumefantrine, the largest malaria product (USD 0.9 bn), sat in the HIV core. Its suppliers include Cipla, Mylan, Macleods and Strides, which all work across disease areas. Moxifloxacin, levofloxacin and delamanid also sat in the HIV core even though they are TB products. Amikacin and meropenem (TB) and older ARVs such as saquinavir and nelfinavir sat with malaria in C3. I read this as a sign that firm capabilities organize the network, not only disease area.

The network pools 2005-2026, and each PQR product name is a separate node, so pediatric and adult forms count as different products.

In [ ]:
# f6 null model
null = pd.read_csv(NET / "null_model_min2.csv")
s = pd.read_csv(ROOT / "outputs" / "tables" / "t8_network_summary.csv").set_index("metric")["value"].astype(float)
fig, axs = plt.subplots(1, 2, figsize=(8.5, 3.4))
for ax, col, obs, lab_ in [(axs[0], "Q", s["modularity_Q"], "Modularity (Q)"),
                           (axs[1], "nmi", s["NMI_area"], "Agreement with disease areas (NMI)")]:
    ax.hist(null[col], bins=25, color="#B4B5B8", edgecolor="white", linewidth=0.8)
    ax.axvline(obs, color=AREA["HIV/ARV"], lw=2)
    ax.text(obs, ax.get_ylim()[1] * 0.92, f"  observed\n  {obs:.2f}", color=INK, fontsize=8.5, va="top")
    ax.set_xlabel(lab_); ax.set_ylabel("Null draws"); ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_xlim(min(null[col].min(), obs) * 0.9 if col == "Q" else 0, obs * 1.25)
axs[0].set_title("Community structure is not an artefact of firm size", loc="left", fontsize=11, fontweight="bold", pad=10)
fig.text(0.01, -0.06, "Gray: 200 re-wired networks that keep each firm's and product's number of links. "
         "Observed values exceed every draw (p < 0.005).", fontsize=7.5, color=MUTED)
fig.tight_layout(); fig.savefig(F / "f6_null_model.png"); plt.show()

In [ ]:
# f7 composition of communities
ct = pd.read_csv(ROOT / "outputs" / "tables" / "t7_communities.csv", index_col=0)
ct = ct[ct.n_products >= 3]
share = ct[list(AREA)].div(ct.n_products, axis=0)[::-1]
names = [COMM_NAME.get(c, c).replace("\n", " ") + f"  (n={int(ct.loc[c, 'n_products'])})" for c in share.index]
fig, ax = plt.subplots(figsize=(8, 2.8)); left = np.zeros(len(share))
for a, col in AREA.items():
    ax.barh(names, share[a] * 100, left=left, color=col, height=0.6, edgecolor="white", linewidth=2, label=a)
    for i, v in enumerate(share[a]):
        if v >= 0.08: ax.text(left[i] + v * 50, i, f"{v:.0%}", ha="center", va="center", color=INK if a == "Malaria" else "white", fontsize=8.5, fontweight="bold")
    left += share[a].values * 100
ax.set_xlim(0, 100); ax.set_xlabel("Share of products in community (%)"); ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.4, 1.0), fontsize=9)
ax.set_title("Communities mostly follow disease areas, with telling exceptions", loc="left", fontsize=11, fontweight="bold", pad=28)
fig.savefig(F / "f7_community_composition.png"); plt.show()
print("ok")